# Pro-bono: Student Reading Campaign Proposal

Author: Engineering Lead (Pro-bono Team)
Date: 2025-09-28

**Executive summary**

This notebook proposes a practical system to increase student reading by combining librarian expertise, an automated recommendation engine (semantic + TF-IDF fallbacks), and campaign-driven nudges surfaced via a simple Gradio UI. It sketches architecture, ingestion, vectorization into Qdrant (or seed file), API endpoints, a Gradio campaign UI (port 7863), monitoring, and rollout estimates. It contains runnable prototypes (guarded) and diagrams suitable for partner review.

## 1) Use case & stakeholders

- Stakeholders: Librarians, Students, School IT, District Admins, Partner NGOs.
- Pain points: Low visibility into what books resonate, limited time for librarians to curate at scale, difficulty measuring impact across schools.
- Goals: Increase student reads by surfacing librarian-approved recommendations, measure campaign impact, and maintain privacy and low-cost deployment.

## Manual demo walkthrough

A companion notebook walks the demo step-by-step, lists the stop files invoked (planner, judge, admin helpers, and student POC), and contains guarded runnable snippets for DB inspection. See: `docs/manual_walkthrough_demo.ipynb`

## Scaling the POC to AWS or GCP

Below is a practical checklist and service mapping for taking this lightweight POC to a production-grade deployment on a major cloud provider (AWS or GCP). The recommendations emphasize low-cost, secure, and observable architecture while supporting model serving, vector search, and audit event persistence.

### High-level architecture
- Frontend: Gradio apps (student & campaign UIs) — run in containers behind a managed ingress/load balancer (ALB/GCLB). Consider converting to a small React app + hosted static site for scale and lower cost; call the API for recommendations.
- API: FastAPI recommendation + campaign endpoints — containerized, autoscaled behind policy-based autoscaling (CPU/RPS). Use managed container services (AWS ECS/Fargate or GCP Cloud Run) for quick ops and lower maintenance.
- Model serving / LLM judge: options depend on model size and latency needs:
  - Hosted LLMs (low ops): OpenAI/Anthropic/Vertex AI/Bedrock — easiest to operate, predictable SLAs, but may have cost/privacy tradeoffs.
  - Managed on-cloud inference (medium ops): AWS Sagemaker Endpoints / GCP Vertex AI Model Serving — supports custom containers and autoscaling, good for larger models and fine-tuning workflows.
  - Self-hosted on GPU instances (higher ops): EC2/GCE with GPUs (e.g., g5, p4, A100) or inference-optimized instances; use Triton or a lightweight HTTP wrapper (Hermes3/llama-server) for the judge. Consider EC2 Spot + autoscaling for cost savings.
- Vector store: production-ready options include Qdrant (self-hosted in containers/VMs) or managed vector DBs such as Pinecone, Weaviate Cloud, or Qdrant Cloud. On AWS/GCP you can also run a HA Qdrant cluster in Kubernetes (EKS/GKE) with persistent volumes.
- Persistent store & auditing: PostgreSQL (RDS / Cloud SQL) for audit_logs/judge_logs, or continue with SQLite for low-volume pilots. Use S3/GCS for artifact storage (catalog exports, model files).
- Observability: Prometheus + Grafana for metrics, Loki or cloud logging for logs, and alerting on error rates, model latency, and judge failures. Use CloudWatch / Stackdriver integrations when using managed services.

### Service mapping (quick)
- AWS: ECS/Fargate or EKS for containers, RDS (Postgres) for DB, S3 for object storage, SageMaker or EC2 GPU for model serving, Elastic Load Balancer / API Gateway for ingress, CloudWatch + X-Ray for telemetry.
- GCP: Cloud Run or GKE for containers, Cloud SQL (Postgres) for DB, Cloud Storage for artifacts, Vertex AI for hosted model serving or GCE GPU instances for self-hosting, Cloud Monitoring / Logging for telemetry.

### Security & compliance
- Use IAM and service accounts with least-privilege. Put the judge/model-serving in a private subnet and only expose a stable, authenticated inference API to the application layer.
- Encrypt data at rest (RDS, S3/GCS) and in transit (TLS everywhere).
- Consider VPC peering / Private Service Connect to keep internal traffic off the public internet when integrating managed model providers or vector DBs.
- For student data, treat PII carefully: anonymize student_id in exports, audit access via role-based controls, and consider a Data Protection Addendum for third-party LLM providers.

### Cost & performance tradeoffs
- Hosted LLMs: pay-per-request, simple ops, but recurring cost may be high for heavy use. Good for early pilots.
- Self-hosted GPU inference: higher fixed infra cost but cheaper per-inference at scale; requires MLOps (model upgrades, monitoring).
- Vector DB choices: managed stores reduce ops burden. Self-hosted Qdrant on k8s is flexible but requires ops resources.

### Rollout checklist (minimal viable to production)
1. Replace SQLite with managed Postgres (RDS/Cloud SQL) and migrate `audit_logs` + `judge_logs`.
2. Containerize Gradio + API (Docker) and deploy to Cloud Run / ECS Fargate with HTTPS endpoint and basic auth or OIDC auth for admin UIs.
3. Choose model-serving approach: start with a hosted LLM provider for judge scoring, then pilot an on-cloud managed model (Vertex AI / SageMaker) if cost/privacy needs change.
4. Move seed catalog and artifacts to S3/GCS; configure CI to upsert vectors to managed vector DB or self-hosted Qdrant cluster.
5. Add monitoring + alerting (errors, judge-empty-responses, high latency, queue backlogs) and log retention policies.
6. Harden access controls (RBAC for Admin UI), enable audit logging, and validate PII handling and data retention policies.
7. Run a 2-week pilot with real librarian workflows, collect metrics (CTR, borrow_rate uplift), and iterate on judge prompts and autos-approval thresholds.

### Example infra cost considerations (very approximate)
- Small pilot (low traffic): $100–$1,000/month using Cloud Run/Fargate + managed DB + hosted LLM usage (light).
- Moderate scale (district): $1k–$10k/month depending on LLM usage; self-hosted inference with spot instances can reduce model costs.
- Large scale (many schools): costs dominated by model serving and vector DB throughput; expect $10k+/month unless using heavy on-prem GPU infra.

### Final notes
- Start with the lowest ops path that meets privacy requirements (hosted LLMs + managed DB + Cloud Run). If costs grow, consider migrating judge/model-serving to managed on-cloud or self-hosted GPU instances with autoscaling and cost controls.
- I can provide a concrete Terraform / CloudFormation starter template and a migration checklist if you want to move forward with either AWS or GCP.


## Tools & tech stack (consolidated)

Below is a concise table that lists the primary tools and technologies, their purpose in the system, and whether they are currently used in this prototype or planned for future use.

| Tool / Tech | Purpose / Use | Status (Current / Future) |
|---|---|---|
| Python 3.10+ (pandas, numpy) | Data munging, notebook analysis, small utility scripts | Current |
| FastAPI + Uvicorn | Recommendation & campaign API endpoints, backend services | Current (sketch) |
| Gradio | Student-facing & campaign UIs used for demos and rapid prototyping | Current (demo) |
| TF-IDF fallback (pure Python) | Lightweight retrieval when embeddings are unavailable; deterministic, low-cost recommender | Current |
| sentence-transformers (optional) | Produce dense embeddings for semantic search and k-NN | Future / Optional (guarded) |
| FAISS / Qdrant | Vector indexes for semantic search; Qdrant recommended for production | Future (Qdrant seed included) |
| Hermes3 (python client + HTTP) | LLM judge: score candidate text, emit structured JSON for audit and auto-approval | Current (preferred judge) / Fallback to heuristic if unavailable |
| Heuristic judge (deterministic) | Deterministic fallback for environments without Hermes3 | Current (fallback) |
| SQLite (agent.db) | Lightweight audit and judge logs storage for pilots | Current |
| Postgres (RDS / Cloud SQL) | Production-grade audit/judge storage with ACID and scaling | Future |
| Qdrant Cloud / Pinecone / Weaviate | Managed vector DB options for scaling vector search | Future |
| Prometheus + Grafana / Cloud Monitoring | Metrics & observability for services, model latency, and judge errors | Future (recommended) |
| S3 / GCS | Artifact and catalog storage (seed files, model artifacts) | Future |
| Terraform / IaC | Infrastructure provisioning for cloud deployments | Future |
| Docker / Containerization | Package and run Gradio, API, and workers reliably | Current (recommended) |


## 8) TF-IDF fallback demo

A lightweight TF-IDF recommender used when embeddings are not available. This demo runs on the small catalog below.

In [ ]:
# Simple TF-IDF fallback recommender
import math, re
from collections import Counter
import pandas as pd
df = pd.read_csv('data/seed_catalog.csv')
def tokenize(s):
    s = s.lower()
    return re.findall(r'\w+', s)
docs = (df['title'] + ' ' + df['description']).tolist()
tf = [Counter(tokenize(d)) for d in docs]
vocab = set().union(*[set(t.keys()) for t in tf])
vocab = sorted(vocab)
idf = {}
N = len(docs)
for w in vocab:
    dfreq = sum(1 for t in tf if w in t)
    idf[w] = math.log((N+1)/(dfreq+1)) + 1
def tfidf_vector(counter):
    return [counter.get(w,0)*idf[w] for w in vocab]
vectors = [tfidf_vector(t) for t in tf]
def cosine(a,b):
    s = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return s/(na*nb) if na and nb else 0
def recommend_tfidf(query, k=3):
    q = Counter(tokenize(query))
    qv = tfidf_vector(q)
    scores = [(i, cosine(qv, v)) for i,v in enumerate(vectors)]
    scores.sort(key=lambda x: -x[1])
    return [(int(df.iloc[i]['id']), df.iloc[i]['title'], round(score,3)) for i,score in scores[:k]]
print('Sample recommendations for ->', recommend_tfidf('space'))

## 9) Embedding mock & Qdrant seed

This cell attempts to build embeddings with sentence-transformers or falls back to deterministic vectors.

In [ ]:
# Build embeddings (mocked if necessary)
import numpy as np, json
from pathlib import Path
texts = []
import pandas as pd
df = pd.read_csv('data/seed_catalog.csv')
texts = (df['title'] + ' ' + df['description']).tolist()
seed_path = Path('docs/qdrant_seed.json')
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb = model.encode(texts, show_progress_bar=False)
    print('Built embeddings with sentence-transformers')
except Exception as e:
    print('sentence-transformers not available or failed, using deterministic random vectors:', e)
    rng = np.random.RandomState(42)
    emb = rng.normal(size=(len(texts), 384))
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
with open(seed_path, 'r') as f:
    seed = json.load(f)
for i, s in enumerate(seed):
    s['vector'] = emb[i].tolist()
with open(seed_path, 'w') as f:
    json.dump(seed, f)
print('Wrote vectors to', seed_path)

## 10) Recommendation Engine Prototype (k-NN queries)

A small k-NN engine using in-memory vectors with optional popularity boosting.

In [ ]:
# In-memory k-NN recommendation using embeddings + borrow_count boosting
import numpy as np, json
from math import exp
seed = json.load(open('docs/qdrant_seed.json'))
vectors = np.array([np.array(s['vector']) for s in seed])
meta = [s['payload'] for s in seed]
borrow_counts = np.array([m.get('borrow_count',0) for m in meta], dtype=float)
def recommend_knn(query_text, top_k=3, popularity_weight=0.3):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        qv = model.encode([query_text])[0]
    except Exception:
        import hashlib
        h = hashlib.sha256(query_text.encode()).digest()
        rng = np.random.RandomState(int.from_bytes(h[:4],'little'))
        qv = rng.normal(size=vectors.shape[1])
        qv = qv / np.linalg.norm(qv)
    sims = vectors.dot(qv)
    pop_scores = (borrow_counts - borrow_counts.min()) / (borrow_counts.ptp() + 1e-9)
    combined = sims * (1 - popularity_weight) + pop_scores * popularity_weight
    idx = np.argsort(-combined)[:top_k]
    return [(meta[i]['id'], meta[i]['title'], float(combined[i])) for i in idx]
print(recommend_knn('space exploration', top_k=3))

## 11) Campaign UI: Gradio app (sketch)

A guarded minimal Gradio Blocks app for campaign building; run locally to test.

In [ ]:
# Minimal Gradio campaign UI (guarded).
try:
    import gradio as gr
    def make_recs(target_age, query):
        return '<br>'.join([f\
 for t in recommend_knn(query or target_age, top_k=5)])
    with gr.Blocks() as demo_app:
        gr.Markdown('## Campaign builder (demo)')
        age = gr.Textbox(label='Target age/grade')
        query = gr.Textbox(label='Context / prompt')
        out = gr.HTML()
        btn = gr.Button('Show weekly recs')
        btn.click(lambda a,q: make_recs(a,q), inputs=[age,query], outputs=out)
    print('Gradio is available. To run the Campaign UI, call: demo_app.launch(server_name=\"0.0.0.0\", server_port=7863, prevent_thread_lock=True)')
except Exception as e:
    print('Gradio not installed or failed to import:', e)

## 12) API Service: FastAPI endpoints (sketch)

A minimal FastAPI sketch for recommendations; run in a separate process for production.

In [ ]:
# FastAPI sketch (do not run inside notebook in production)
try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    app = FastAPI()
    class RecommendReq(BaseModel):
        student_id: str = None
        context: str = None
    @app.post('/recommend')
    async def recommend(req: RecommendReq):
        recs = recommend_knn(req.context or 'general', top_k=5)
        return {'recommendations': [{'id': r[0], 'title': r[1], 'score': r[2]} for r in recs]}
    print('FastAPI sketch loaded (run with uvicorn in a separate process)')
except Exception as e:
    print('FastAPI not available in this environment:', e)

## 13) Hermes3 judge: prompt tuning & demo (runnable)

This section describes the prompt contract and includes a guarded demo cell that runs the planner, invokes the judge scaffold, and shows Admin UI output. The judge code in the repo (`tools/llm_judge.py`) includes guarded logic to prefer Hermes3 (python client) then HTTP, and falls back to a heuristic.

In [ ]:
# Demo flow: run the agent job to create recommendations, let the judge score them, and show Admin UI output.
# Guarded: won't crash if optional services (Hermes3) are missing.
import subprocess, os, json, sqlite3
ROOT = os.path.abspath('')
try:
    print('Running agent_job.py to plan and persist recommendations...')
    ret = subprocess.run([os.path.join('.venv','bin','python'), 'agentic-rag-mvp/tools/agent_job.py'], capture_output=True, text=True)
    print('agent_job output (tail):')
    print(ret.stdout[-800:])
    if ret.returncode != 0:
        print('agent_job exited with code', ret.returncode)
        print(ret.stderr[:800])
except Exception as e:
    print('Could not run agent_job.py via subprocess:', e)

# Show the latest pending rows and any associated judge_logs via admin helper
try:
    from agentic_rag_mvp.tools import admin_ui
    rows = admin_ui.get_pending_rows()
    print('Pending audit rows (id,action_type,student_id,book_title,judge_label,judge_score):')
    for r in rows[:20]:
        print(r['id'], r['action_type'], r.get('student_id'), r.get('book_title'), r.get('judge_label'), r.get('judge_score'))
    if rows:
        first = rows[0]['id']
        print('Full judge JSON for first pending id:', first)
        print(admin_ui.fetch_full_judge_json(first) or '(no judge row)')
except Exception as e:
    print('Could not import admin_ui or fetch pending rows:', e)

# Quick DB query as fallback
try:
    DB = os.path.join('agentic-rag-mvp', 'data', 'agent.db')
    if os.path.exists(DB):
        con = sqlite3.connect(DB)
        cur = con.cursor()
        cur.execute("SELECT id, action_type, payload, status FROM audit_logs ORDER BY id DESC LIMIT 10")
        for row in cur.fetchall():
            print('AUDIT:', row[0], row[1], str(row[2])[:120], row[3])
        cur.execute("SELECT id, audit_id, model, score, label, substr(reason,1,140), created_at FROM judge_logs ORDER BY created_at DESC LIMIT 10")
        for row in cur.fetchall():
            print('JUDGE:', row)
        con.close()
    else:
        print('DB not found at', DB)
except Exception as e:
    print('DB query failed:', e)

## 14) Reference architecture diagram (detailed)

Below is a more detailed reference architecture highlighting cloud services and layers.

```mermaid
graph LR
  Browser["Student Browser / Gradio UI"] -->|HTTPS| ALB["Load Balancer / CDN"]
  ALB -->|route| API["FastAPI (Cloud Run / ECS)"]
  API -->|recommend| Vector["Qdrant (managed or self-hosted)"]
  API -->|audit| DB["Postgres (RDS / Cloud SQL)"]
  API -->|enqueue| JudgeQueue["Judge queue (SQS / PubSub)"]
  JudgeQueue -->|worker| JudgeWorker["Judge workers (Fargate / Cloud Run)"]
  JudgeWorker -->|call| Model["Model serving (Hosted LLMs / SageMaker / Vertex / EC2 GPU)"]
  Model -->|response| JudgeWorker --> DB
  Admin["Admin UI"] -->|auth| ALB --> API --> DB
  Storage[S3/GCS] -->|catalog artifacts| API
  Monitoring["Cloud Monitoring / Prometheus"] --> API & Model & Vector & DB
```

## 16) Appendix: runbook snippets & commands

- Start Hermes3 (local helper): `agentic-rag-mvp/scripts/run_hermes3.sh`
- Run the planner manually: `.venv/bin/python agentic-rag-mvp/tools/agent_job.py`
- Inspect judge_logs: `sqlite3 agentic-rag-mvp/data/agent.db "SELECT * FROM judge_logs ORDER BY created_at DESC LIMIT 20;"`
- Run the notebook manual test cell to exercise the planner/judge flow (guarded).